# Phase 2: Data Cleaning & Preprocessing — Telco Customer Churn

## 1. Overview & Objectives
In **Phase 2**, we transform the raw dataset into a clean, well-formatted, and robust state ready for exploratory data analysis (EDA) and feature engineering.

### Pipeline Checklist:
- [x] Inspect explicit and hidden missing values (`TotalCharges` whitespace strings).
- [x] Convert `TotalCharges` to `float64` and impute missing charges with `0.0` (validated by `tenure == 0`).
- [x] Standardize redundant service categories (`No internet service` and `No phone service` $\to$ `No`).
- [x] Encode the target variable `Churn` from `Yes`/`No` to binary integers `1`/`0`.
- [x] Validate dataset integrity (confirm 0 missing values, 0 duplicate records).
- [x] Export cleaned dataset to `data/processed/cleaned_churn_data.csv`.
- [x] Verify integration with production module `src.preprocessing.clean_churn_data`.

In [ ]:
# 2. Library Imports
import pandas as pd
import numpy as np
from pathlib import Path

# Configure pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

In [2]:
# 3. Load Raw Dataset
raw_path = Path('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df_raw = pd.read_csv(raw_path)
print(f"Raw dataset loaded successfully: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns.")
df_raw.head(3)

Raw dataset loaded successfully: 7043 rows, 21 columns.


## 4. Resolving TotalCharges Type & Imputing Missing Values
As uncovered in Phase 1, `TotalCharges` is stored as an `object` due to 11 records containing whitespace characters (`' '`). Let's inspect these records.

In [3]:
# Identify whitespace rows
blank_mask = df_raw['TotalCharges'].astype(str).str.strip() == ''
print(f"Number of blank TotalCharges records: {blank_mask.sum()}")

# Inspect tenure and churn status for blank records
blank_records = df_raw[blank_mask][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Contract', 'Churn']]
print("\nRecords with blank TotalCharges:")
print(blank_records)

Number of blank TotalCharges records: 11

Records with blank TotalCharges:
      customerID  tenure  MonthlyCharges TotalCharges  Contract Churn
488   4472-LVYGI       0           52.55               Two year    No
753   3115-CZMZD       0           20.25               Two year    No
936   5709-LVOEQ       0           80.85               Two year    No
1082  4367-NUYAO       0           25.75               Two year    No
1340  1371-DWPAZ       0           56.05               Two year    No
3331  7644-OMVMY       0           19.85               Two year    No
3826  3213-VVOLG       0           25.35               Two year    No
4380  2520-SGTTA       0           20.00               Two year    No
5218  2923-ARZLG       0           19.70               One year    No
6670  4075-WKNIU       0           73.35               Two year    No
6754  2775-SEFEE       0           61.90               Two year    No


In [4]:
# Clean TotalCharges: convert to float64, filling NaN (from tenure==0) with 0.0
df_clean = df_raw.copy()
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'].astype(str).str.strip(), errors='coerce').fillna(0.0)

print(f"TotalCharges dtype after conversion: {df_clean['TotalCharges'].dtype}")
print(f"Remaining null values in TotalCharges: {df_clean['TotalCharges'].isnull().sum()}")
print(f"TotalCharges summary for tenure == 0:\n{df_clean[df_clean['tenure'] == 0]['TotalCharges'].value_counts()}")

TotalCharges dtype after conversion: float64
Remaining null values in TotalCharges: 0
TotalCharges summary for tenure == 0:
TotalCharges
0.0    11
Name: count, dtype: int64


## 5. Standardizing Redundant Sub-Service Categories
Several columns contain redundant categories like `'No internet service'` or `'No phone service'`. Standardizing these to `'No'` simplifies the feature space, reduces collinearity during one-hot encoding, and retains 100% of the information since `InternetService` and `PhoneService` already indicate service status.

In [5]:
# Categories to standardize
service_cols = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]

print("Before standardization:")
for col in service_cols[:2]:
    print(f"{col}: {df_clean[col].unique().tolist()}")

# Replace redundant sub-categories
for col in service_cols:
    df_clean[col] = df_clean[col].replace({'No internet service': 'No'})

df_clean['MultipleLines'] = df_clean['MultipleLines'].replace({'No phone service': 'No'})

print("\nAfter standardization:")
for col in service_cols[:2]:
    print(f"{col}: {df_clean[col].unique().tolist()}")
print(f"MultipleLines: {df_clean['MultipleLines'].unique().tolist()}")

Before standardization:
OnlineSecurity: ['No', 'Yes', 'No internet service']
OnlineBackup: ['Yes', 'No', 'No internet service']

After standardization:
OnlineSecurity: ['No', 'Yes']
OnlineBackup: ['Yes', 'No']
MultipleLines: ['No', 'Yes']


## 6. Target Variable Encoding (`Churn`)
Encode `Churn` as binary integer: `No` $\to 0$, `Yes` $\to 1$.

In [6]:
# Map Churn to 0 and 1
df_clean['Churn'] = df_clean['Churn'].map({'Yes': 1, 'No': 0})

churn_summary = pd.DataFrame({
    'Count': df_clean['Churn'].value_counts(),
    'Percentage (%)': (df_clean['Churn'].value_counts(normalize=True) * 100).round(2)
})
print("Cleaned Target Distribution:")
print(churn_summary)

Cleaned Target Distribution:
       Count  Percentage (%)
Churn                       
0       5174           73.46
1       1869           26.54


## 7. Data Quality & Integrity Validation

In [7]:
# 1. Check for missing values across the entire dataset
total_nulls = df_clean.isnull().sum().sum()
print(f"Total null values in dataset: {total_nulls}")

# 2. Check for duplicate rows
duplicate_rows = df_clean.duplicated().sum()
print(f"Duplicate rows in dataset: {duplicate_rows}")

# 3. Check data types
print("\nFinal Data Types:")
print(df_clean.dtypes)

# 4. Summary statistics of clean numerical features
print("\nNumerical Features Summary:")
print(df_clean[['tenure', 'MonthlyCharges', 'TotalCharges']].describe().round(2).T)

Total null values in dataset: 0
Duplicate rows in dataset: 0

Final Data Types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object

Numerical Features Summary:
                 count     mean      std    min     25%      50%      75%      max
tenure          7043.0    32.37    24.56   0.00    9.00    29.00    55.00    72.00
MonthlyCharges  7043.0    64.76    30.09  18.25   35.50    70.35    89.85   118.75
TotalCharges    7043.0  2279.73  2266.79

## 8. Exporting the Cleaned Dataset

In [8]:
processed_path = Path('../data/processed/cleaned_churn_data.csv')
processed_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(processed_path, index=False)

print(f"Cleaned dataset saved to: {processed_path.resolve()}")
print(f"File size: {processed_path.stat().st_size:,} bytes")
print(f"Row count: {len(df_clean)}, Column count: {len(df_clean.columns)}")

Cleaned dataset saved to: D:\ML\customer-churn-prediction\data\processed\cleaned_churn_data.csv
File size: 804,843 bytes
Row count: 7043, Column count: 21


## 9. Verification with Production Pipeline (`src.preprocessing`)
Verify that our modular function `clean_churn_data` in `src/preprocessing.py` matches the notebook operations exactly.

In [9]:
import sys
sys.path.append('..')
from src.preprocessing import clean_churn_data, load_raw_data

module_df_clean = clean_churn_data(load_raw_data())
pd.testing.assert_frame_equal(df_clean, module_df_clean)
print("SUCCESS: Production module src.preprocessing.clean_churn_data matches notebook operations perfectly!")

SUCCESS: Production module src.preprocessing.clean_churn_data matches notebook operations perfectly!
